In [2]:

import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sb 

In [3]:
## Loading the agriculture dataset 
df = pd.read_csv("../data/preprocessed/cleaned_agridata.csv")

In [4]:
df.head()

,Unnamed: 0,State,District,Crop,Season,Area,Production,Production Units,Yield,Year_end
0,0,Andaman and Nicobar Islands,NICOBARS,Arecanut,Kharif,1254.0,2061.0,Tonnes,1.643541,2002
1,1,Andaman and Nicobar Islands,NICOBARS,Arecanut,Whole Year,1258.0,2083.0,Tonnes,1.655803,2003
2,2,Andaman and Nicobar Islands,NICOBARS,Arecanut,Whole Year,1261.0,1525.0,Tonnes,1.209358,2004
3,3,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,Arecanut,Kharif,3100.0,5239.0,Tonnes,1.690000,2002
4,4,Andaman and Nicobar Islands,SOUTH ANDAMANS,Arecanut,Whole Year,3105.0,5267.0,Tonnes,1.696296,2003


## Removing the production and production units column column 

In [5]:
df = df.drop(columns="Production Units")

In [6]:
df = df.drop(columns="Production")

In [7]:
df = df.drop(columns="Unnamed: 0")

In [8]:
df.head()

,State,District,Crop,Season,Area,Yield,Year_end
0,Andaman and Nicobar Islands,NICOBARS,Arecanut,Kharif,1254.0,1.643541,2002
1,Andaman and Nicobar Islands,NICOBARS,Arecanut,Whole Year,1258.0,1.655803,2003
2,Andaman and Nicobar Islands,NICOBARS,Arecanut,Whole Year,1261.0,1.209358,2004
3,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,Arecanut,Kharif,3100.0,1.690000,2002
4,Andaman and Nicobar Islands,SOUTH ANDAMANS,Arecanut,Whole Year,3105.0,1.696296,2003


## Train Test Split

In [9]:
train = df[(df['Year_end']>=1998) & (df['Year_end']<= 2017)]
test = df[(df['Year_end']>=2018) & (df['Year_end']<= 2020)]

X_train = train.drop('Yield', axis=1)
y_train = train['Yield']

X_test = test.drop('Yield', axis=1)
y_test = test['Yield']


## Season Data Cleaning 

In [10]:
df['Season'].unique()

array(['Kharif', 'Whole Year', 'Rabi', 'Autumn', 'Summer', 'Winter'],
      dtype=object)

In [11]:
df['Season'].value_counts()


Season
Kharif        136165
Rabi           99805
Whole Year     67265
Summer         21974
Winter          8238
Autumn          6967
Name: count, dtype: int64

In [12]:
## Whole year represents aggregated annual data and 
# may behave differently from seasonal records 

### Log transformation 
  1. Reducing the skewness 
  2. Reducing the impact of Outliers 
  

In [12]:
df['Area'] = np.log1p(df['Area'])
df['Yield'] = np.log1p(df['Yield'])

In [14]:
### Standard Scaling and encoding of the data with pipeline 


In [13]:
from sklearn.preprocessing import StandardScaler 
scaler = StandardScaler()

In [14]:
! pip install category_encoders


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
## Defining columns for 
numeric_cols = ['Area','Year_end']
ohe_cols = ['Crop','Season','State']
target_enc_cols = ['District']

In [16]:
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline 
from category_encoders.target_encoder import TargetEncoder 

In [17]:
## Build preprocessor 
preprocessor = ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),numeric_cols),
        ('ohe',OneHotEncoder(handle_unknown='ignore'),ohe_cols),
        ('target',TargetEncoder(),target_enc_cols)
    ]
)

In [18]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso 
from sklearn.ensemble import RandomForestRegressor 

In [19]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

models = {
    "Linear Regression": LinearRegression(),

    "Ridge": Ridge(alpha=1.0),

    "Lasso": Lasso(alpha=0.01, max_iter=5000),

    "Random Forest": RandomForestRegressor(
        n_estimators=50,
        max_depth=15,
        n_jobs=-1,
        verbose=1,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        n_jobs=-1,
        verbosity=1,
        random_state=42
    )
}


## Cross Validation 
 1. Helps to find the best model based on how cv scores for the test sets.
 2. Reduces variance
 3. Give Stability insight
 4. More reliable model comparison

In [22]:
from sklearn.model_selection import cross_val_score 
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

results = {}

best_model = None;
best_r2 = -1;

for name, model in models.items():
    
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2')


    mean_r2 = cv_scores.mean()
    std_r2 = cv_scores.std()
    
    results[name] = {
        "CV Mean R2": mean_r2,
        "CV Std R2": std_r2
    }

    if mean_r2>best_r2:
       best_r2 = mean_r2
       best_model = pipe
    
    
    
results

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   51.8s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   59.2s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done  50 out of  50 | elapsed:    0.1s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   49.9s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:   57.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done  50 out of  50 | elapsed:    0.1s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   53.0s
[Parall

{'Linear Regression': {'CV Mean R2': np.float64(0.7845984770453702),
  'CV Std R2': np.float64(0.03781436337427209)},
 'Ridge': {'CV Mean R2': np.float64(0.7843087383107024),
  'CV Std R2': np.float64(0.03765363440252355)},
 'Lasso': {'CV Mean R2': np.float64(0.7846306676435928),
  'CV Std R2': np.float64(0.03778373283407848)},
 'Random Forest': {'CV Mean R2': np.float64(0.8988183270823645),
  'CV Std R2': np.float64(0.01899148417235741)},
 'XGBoost': {'CV Mean R2': np.float64(0.8960024949157475),
  'CV Std R2': np.float64(0.019088608703892108)}}

In [23]:
best_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Area', 'Year_end']),
                                                 ('ohe',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Crop', 'Season', 'State']),
                                                 ('target', TargetEncoder(),
                                                  ['District'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=15, n_estimators=50, n_jobs=-1,
                                       random_state=42, verbose=1))])

In [24]:
best_r2

np.float64(0.8988183270823645)

## Hyperparameter tuning with RandomSearchCV

In [25]:
param_dist = {
    'regressor__n_estimators': [100, 150 ],
    'regressor__max_depth': [10, 15, 20, None],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4],
    'regressor__max_features': ['sqrt', 'log2']
}

In [26]:
from sklearn.model_selection import RandomizedSearchCV 

random_search = RandomizedSearchCV(
    estimator = best_model,
    param_distributions = param_dist,
    n_iter = 15,
    cv=3,
    scoring = 'r2',
    n_jobs=-1,
    verbose = 2,
    random_state = 42
)

In [27]:
random_search.fit(X_train,y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   14.9s
[Parallel(n_jobs=-1)]: Done 150 out of 150 | elapsed:   51.7s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.3s
[Parallel(n_jobs=4)]: Done 150 out of 150 | elapsed:    1.0s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   14.3s
[Parallel(n_jobs=-1)]: Done 150 out of 150 | elapsed:   50.7s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks  

[CV] END regressor__max_depth=None, regressor__max_features=sqrt, regressor__min_samples_leaf=2, regressor__min_samples_split=5, regressor__n_estimators=150; total time=34.9min
[CV] END regressor__max_depth=15, regressor__max_features=log2, regressor__min_samples_leaf=1, regressor__min_samples_split=5, regressor__n_estimators=100; total time= 1.8min
[CV] END regressor__max_depth=10, regressor__max_features=sqrt, regressor__min_samples_leaf=4, regressor__min_samples_split=2, regressor__n_estimators=100; total time=  30.1s
[CV] END regressor__max_depth=10, regressor__max_features=sqrt, regressor__min_samples_leaf=4, regressor__min_samples_split=2, regressor__n_estimators=100; total time=  28.5s
[CV] END regressor__max_depth=None, regressor__max_features=log2, regressor__min_samples_leaf=2, regressor__min_samples_split=2, regressor__n_estimators=100; total time=28.0min
[CV] END regressor__max_depth=20, regressor__max_features=log2, regressor__min_samples_leaf=1, regressor__min_samples_spl

[Parallel(n_jobs=-1)]: Done 150 out of 150 | elapsed: 34.3min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.5s
[Parallel(n_jobs=4)]: Done 150 out of 150 | elapsed:    1.6s finished
[Parallel(n_jobs=-1)]: Done 150 out of 150 | elapsed: 32.3min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.4s
[Parallel(n_jobs=4)]: Done 150 out of 150 | elapsed:    1.5s finished
[Parallel(n_jobs=-1)]: Done 150 out of 150 | elapsed: 33.7min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done 150 out of 150 | elapsed:    0.7s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.


[CV] END regressor__max_depth=None, regressor__max_features=sqrt, regressor__min_samples_leaf=2, regressor__min_samples_split=5, regressor__n_estimators=150; total time=35.4min
[CV] END regressor__max_depth=15, regressor__max_features=log2, regressor__min_samples_leaf=1, regressor__min_samples_split=5, regressor__n_estimators=100; total time= 1.8min
[CV] END regressor__max_depth=10, regressor__max_features=sqrt, regressor__min_samples_leaf=4, regressor__min_samples_split=2, regressor__n_estimators=100; total time=  29.7s
[CV] END regressor__max_depth=None, regressor__max_features=log2, regressor__min_samples_leaf=2, regressor__min_samples_split=2, regressor__n_estimators=100; total time=30.8min
[CV] END regressor__max_depth=None, regressor__max_features=log2, regressor__min_samples_leaf=2, regressor__min_samples_split=2, regressor__n_estimators=150; total time=34.3min
[CV] END regressor__max_depth=10, regressor__max_features=log2, regressor__min_samples_leaf=1, regressor__min_samples_s

[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:  5.5min
[Parallel(n_jobs=-1)]: Done 150 out of 150 | elapsed: 18.7min finished


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               StandardScaler(),
                                                                               ['Area',
                                                                                'Year_end']),
                                                                              ('ohe',
                                                                               OneHotEncoder(handle_unknown='ignore'),
                                                                               ['Crop',
                                                                                'Season',
                                                                                'State']),
                                                                              ('target',
                                                                               TargetEncoder(),
                                                                               ['District'])])),
                                             ('regressor',
                                              RandomForestRegressor(max_depth=15,
                                                                    n_estimators=50,
                                                                    n_jobs=-1,
                                                                    random_state=42,
                                                                    verbose=1))]),
                   n_iter=15, n_jobs=-1,
                   param_distributions={'regressor__max_depth': [10, 15, 20,
                                                                 None],
                                        'regressor__max_features': ['sqrt',
                                                                    'log2'],
                                        'regressor__min_samples_leaf': [1, 2,
                                                                        4],
                                        'regressor__min_samples_split': [2, 5,
                                                                         10],
                                        'regressor__n_estimators': [100, 150]},
                   random_state=42, scoring='r2', verbose=2)

In [28]:
random_search.best_params_


{'regressor__n_estimators': 150,
 'regressor__min_samples_split': 5,
 'regressor__min_samples_leaf': 2,
 'regressor__max_features': 'sqrt',
 'regressor__max_depth': None}

In [28]:


rf = RandomForestRegressor(
    n_estimators=150,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

final_model = Pipeline([
    ('preprocessor', preprocessor),  
    ('regressor', rf)
])

In [29]:
final_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Area', 'Year_end']),
                                                 ('ohe',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Crop', 'Season', 'State']),
                                                 ('target', TargetEncoder(),
                                                  ['District'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=12, max_features='sqrt',
                                       min_samples_leaf=2, min_samples_split=5,
                                       n_estimators=150, n_jobs=-1,
                                       random_state=42))])

In [30]:
final_model.score(X_test, y_test)


0.8000076584254782

In [31]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

y_pred = final_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Test R2:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

Test R2: 0.8000076584254782
MAE: 41.05756101135454
RMSE: 431.5633728554083


In [32]:
train_r2 = final_model.score(X_train, y_train)
print("Train R2:", train_r2)
print("Test R2:", r2)

Train R2: 0.8581144299341205
Test R2: 0.8000076584254782


In [33]:
importances = final_model.named_steps['regressor'].feature_importances_

feature_names = final_model.named_steps['preprocessor'].get_feature_names_out()

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

feature_importance_df.head(20)

,Feature,Importance
11,ohe__Crop_Coconut,0.736484
99,target__District,0.058142
0,num__Area,0.043612
62,ohe__Season_Whole Year,0.034756
1,num__Year_end,0.018615
98,ohe__State_West Bengal,0.017904
60,ohe__Season_Rabi,0.009506
59,ohe__Season_Kharif,0.008346
93,ohe__State_Tamil Nadu,0.007442
89,ohe__State_Puducherry,0.006914


In [34]:
y_pred = final_model.predict(X_test)

residuals = y_test - y_pred
residuals

221009    -5.225550
221010    -5.308406
221012    -7.678561
221013    -7.577808
221015     1.020656
            ...    
293419   -16.598884
293420   -22.051873
293421   -15.456673
293422   -28.315418
293423   -29.586941
Name: Yield, Length: 54834, dtype: float64

In [35]:
import joblib
joblib.dump(final_model, "final_model.pkl")


['final_model.pkl']